Scrap Selenium 

Idea es hacer scraping a 
afrofy.com.ar  obtener la serie historica de precios de la soja 

instalar el geclodriver.exe ubicacion del navegador 


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# Configuración de opciones de Chrome
options = Options()
# (Opcional) Podés agregar argumentos, por ejemplo, para ejecutar sin GUI:
# options.add_argument("--headless")

# WebDriver Manager se encarga de descargar y configurar el chromedriver adecuado
service = Service(ChromeDriverManager().install())

# Crear el navegador usando Chrome
driver = webdriver.Chrome(service=service, options=options)

# Ir a la web deseada
driver.get("https://news.agrofy.com.ar/granos/precios-pizarra")
print("Página abierta y navegada correctamente")

# (Opcional) Esperar para visualizar el sitio
#time.sleep(5)

# Cerrar el navegador
#driver.quit()

# Espera hasta que el campo de fecha sea visible (busca un input de tipo text o date)

Página abierta y navegada correctamente


In [9]:
from selenium.webdriver.support.ui import WebDriverWait

##WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, "//h3[contains(text(), 'Rosario')]"))    )

##table_containers = driver.find_elements(By.CLASS_NAME, "nav-item nav-link active") ##"table-agrofy is-gray table")


table_containers = driver.find_elements(By.CLASS_NAME, "container") ##"table-agrofy is-gray table")



if not table_containers:
        print("No se encontraron contenedores de tablas de precios en la página.")
else:
        print(f"Se encontraron {len(table_containers)} secciones de tablas de precios.")
        all_pizarra_data = {}

        for i, container in enumerate(table_containers):
            
            print(f"Procesando contenedor {i + 1} de {container}...")


        print("\n--- Extracción completa de todas las tablas de pizarra ---")
        # Aquí puedes acceder a los datos de cada localidad:
        # print(all_pizarra_data['Rosario'])
        # print(all_pizarra_data['Buenos Aires'])





Se encontraron 8 secciones de tablas de precios.
Procesando contenedor 1 de <selenium.webdriver.remote.webelement.WebElement (session="ca0238b5e5417d5881f75528e5ac2ac5", element="f.49C897B2F90294E75CC62A7D1B13D1EC.d.E1C039606114E892535FB79CAB4D3494.e.33")>...
Procesando contenedor 2 de <selenium.webdriver.remote.webelement.WebElement (session="ca0238b5e5417d5881f75528e5ac2ac5", element="f.49C897B2F90294E75CC62A7D1B13D1EC.d.E1C039606114E892535FB79CAB4D3494.e.345")>...
Procesando contenedor 3 de <selenium.webdriver.remote.webelement.WebElement (session="ca0238b5e5417d5881f75528e5ac2ac5", element="f.49C897B2F90294E75CC62A7D1B13D1EC.d.E1C039606114E892535FB79CAB4D3494.e.346")>...
Procesando contenedor 4 de <selenium.webdriver.remote.webelement.WebElement (session="ca0238b5e5417d5881f75528e5ac2ac5", element="f.49C897B2F90294E75CC62A7D1B13D1EC.d.E1C039606114E892535FB79CAB4D3494.e.347")>...
Procesando contenedor 5 de <selenium.webdriver.remote.webelement.WebElement (session="ca0238b5e5417d5881

In [10]:
from selenium.webdriver.support.ui import WebDriverWait

##WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, "//h3[contains(text(), 'Rosario')]"))    )

##table_containers = driver.find_elements(By.CLASS_NAME, "nav-item nav-link active") ##"table-agrofy is-gray table")


table_containers = driver.find_elements(By.CLASS_NAME, "container") ##"table-agrofy is-gray table")



if not table_containers:
        print("No se encontraron contenedores de tablas de precios en la página.")
else:
        print(f"Se encontraron {len(table_containers)} secciones de tablas de precios.")
        all_pizarra_data = {}

        for i, container in enumerate(table_containers):
            
            print(f"Procesando contenedor {i + 1} de {container}...")

            try:
                # Extraer el título de la localidad (ej. Rosario, Buenos Aires)
                location_element = container.find_element(By.TAG_NAME, "h3")
                location_name = location_element.text.strip()
                print(f"\n--- Procesando tabla para: {location_name} ---")

                # Encontrar la tabla dentro de este contenedor
                table = container.find_element(By.TAG_NAME, "table")

                # Extraer encabezados de la tabla (thead)
                headers = []
                header_elements = table.find_elements(By.TAG_NAME, "th")
                if not header_elements:
                    # A veces los encabezados están en tr/tds dentro de thead sin th
                    header_elements = table.find_elements(By.XPATH, ".//thead/tr/td")

                for header_el in header_elements:
                    headers.append(header_el.text.strip())

                # Si no se encontraron encabezados, usar un valor predeterminado o saltar
                if not headers:
                    print(f"Advertencia: No se encontraron encabezados para {location_name}. Usando encabezados predeterminados.")
                    headers = ["Producto", "Precio", "Variación"] # Encabezados comunes

                # Extraer filas de datos (tbody)
                rows = []
                data_rows = table.find_elements(By.TAG_NAME, "tr")
                for row_el in data_rows:
                    cols = row_el.find_elements(By.TAG_NAME, "td")
                    if cols: # Asegurarse de que sea una fila de datos y no un encabezado
                        row_data = [col.text.strip() for col in cols]
                        rows.append(row_data)

                # Crear un DataFrame de pandas
                if rows:
                    df = pd.DataFrame(rows, columns=headers[:len(rows[0])]) # Ajustar columnas si hay más datos que encabezados
                    print(df.to_string(index=False)) # Imprimir el DataFrame sin el índice
                    all_pizarra_data[location_name] = df
                else:
                    print(f"No se encontraron datos en la tabla para {location_name}.")

            except Exception as e:
                print(f"Error al procesar una tabla en el contenedor {i}: {e}")

        print("\n--- Extracción completa de todas las tablas de pizarra ---")
        # Aquí puedes acceder a los datos de cada localidad:
        # print(all_pizarra_data['Rosario'])
        # print(all_pizarra_data['Buenos Aires'])





Se encontraron 8 secciones de tablas de precios.
Procesando contenedor 1 de <selenium.webdriver.remote.webelement.WebElement (session="ca0238b5e5417d5881f75528e5ac2ac5", element="f.49C897B2F90294E75CC62A7D1B13D1EC.d.E1C039606114E892535FB79CAB4D3494.e.33")>...
Error al procesar una tabla en el contenedor 0: Message: no such element: Unable to locate element: {"method":"tag name","selector":"h3"}
  (Session info: chrome=137.0.7151.104); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x0x973b03+62899]
	GetHandleVerifier [0x0x973b44+62964]
	(No symbol) [0x0x7a10f3]
	(No symbol) [0x0x7e980e]
	(No symbol) [0x0x7e9bab]
	(No symbol) [0x0x7def51]
	(No symbol) [0x0x80e554]
	(No symbol) [0x0x7dee74]
	(No symbol) [0x0x80e784]
	(No symbol) [0x0x82fd81]
	(No symbol) [0x0x80e306]
	(No symbol) [0x0x7dd670]
	(No symbol) [0x0x7de4e4]
	GetHandleVerifier [0x0xbd4793+2556483]
	G